<blank>

# **Prove di random forest con PCA**

## Preparazione dei dati

---

In [1]:
import numpy as np
# Segnale su cui effettuare Train e Validation
g_0_b = np.loadtxt('data/g_0_signal_b.txt')
g_1_b = np.loadtxt('data/g_1_signal_b.txt')
# Segnale su cui effettuare Test Esterno
g_0_a = np.loadtxt('data/g_0_signal_a.txt')
g_1_a = np.loadtxt('data/g_1_signal_a.txt')

Concatenazione gruppi e vettore dei label:

In [2]:
N0, N1 = g_0_b.shape[0], g_1_b.shape[0]
# concateno g_0 e g_1
signal_b = np.vstack((g_0_b, g_1_b))
signal_a = np.vstack((g_0_a, g_1_a))

# vettore delle risposte
labels = np.concatenate((np.zeros(N0), np.ones(N1))) # uguale per A e B

## Primissima prova

---

In [3]:
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

from sklearn.ensemble import RandomForestClassifier as RFC

In [4]:
pipe = Pipeline([
    ("scaling", MinMaxScaler()),       
    ("reduce_dim", PCA(n_components=5, random_state=42)),
    ("classify", RFC(random_state=42)) 
])

pipe.fit(signal_b, labels)
alt_accuracy = pipe.score(signal_a, labels)

print(f"L'accuracy del random forest è: {alt_accuracy:.4f}")

L'accuracy del random forest è: 0.5000


<blank>

# GridSearch

---

In [5]:
rkf = RepeatedStratifiedKFold(n_splits=8, n_repeats=10, random_state=42)

# PCA
N_COMPONENTS_OPTIONS = [5, 15, 20]
# ESTIMATOR
N_ESTIMATOR_OPTIONS = [100, 500, 1000]
OBJ_FUNCTION_OPTIONS = ['gini', 'entropy']
MAX_FEATURES_OPTIONS = ["sqrt", "log2"]
MAX_DEPTH_OPTIONS = [None, 3, 5]
MIN_SAMPLES_LEAF_OPTIONS = [1, 2, 4]


# 1. Definizione Pipeline #
pipe = Pipeline([
    # Step 1: Scaling
    ("scaling", StandardScaler()),       
    
    # Step 2: Riduzione dimensionalità (PCA)
    ("reduce_dim", PCA(random_state=42)),
    
    # Step 3: Classificatore
    ("classify", RFC(bootstrap=True)) 
])

# 2. Definizione griglia dei parametri #
param_grid = {
    
    # Per provare parametri specifici di uno step (PCA)
    "reduce_dim__n_components": N_COMPONENTS_OPTIONS, 
    
    # Per provare parametri del classificatore
    "classify__n_estimators": N_ESTIMATOR_OPTIONS,
    
    # Per provare diverse objective funcions
    "classify__criterion": OBJ_FUNCTION_OPTIONS,
    
    # Diversi max features
    "classify__max_features": MAX_FEATURES_OPTIONS,
    
    "classify__max_depth": MAX_DEPTH_OPTIONS, # Ferma la crescita dell'albero oltre un certo limite
    
    "classify__min_samples_leaf": MIN_SAMPLES_LEAF_OPTIONS # Stabilizza le foglie
}

# 3. Configurazione GridSearch #
grid = GridSearchCV(
    pipe, 
    param_grid=param_grid, 
    cv=rkf,
    n_jobs=-1, # «Number of jobs to run in parallel. -1 means using all processors»
    scoring='accuracy'
)

# 4. Training e Validation (su Segnale B) #
grid.fit(signal_b, labels)

# 5. Risultati #
print(f"La miglior configurazione: {grid.best_params_}")
print(f"Fornisce accuracy in validation: {grid.best_score_:.4f}")

# 6. Test su segnale A #
accuracy_finale = grid.score(signal_a, labels)
print(f"Risultato sul set indipendente (Segnale A): {accuracy_finale:.4f}")

La miglior configurazione: {'classify__criterion': 'entropy', 'classify__max_depth': 3, 'classify__max_features': 'sqrt', 'classify__min_samples_leaf': 1, 'classify__n_estimators': 500, 'reduce_dim__n_components': 15}
Fornisce accuracy in validation: 0.6604
Risultato sul set indipendente (Segnale A): 0.4545


In [6]:
import pandas as pd

results_df = pd.DataFrame(grid.cv_results_)
# Ciascuna combinazione di parametri è una riga
print(f"Numero totale di configurazioni provate: {results_df.shape[0]}")

# Selezioniamo solo le colonne interessanti per pulire la vista
columns_to_show = [
    'param_reduce_dim__n_components',
    'param_classify__n_estimators', 
    'param_classify__criterion', 
    'param_classify__max_features', 
    'param_classify__max_depth',
    'param_classify__min_samples_leaf',
    'mean_test_score', 
    'std_test_score', 
    'rank_test_score'
]

# Ordiniamo per classifica (rank_test_score)
analysis = results_df[columns_to_show].sort_values('rank_test_score')

# Se si ha, è comodo aprire analysis in un viewer tipo Data Wrangler
analysis.head(20)

Numero totale di configurazioni provate: 324


,param_reduce_dim__n_components,param_classify__n_estimators,param_classify__criterion,param_classify__max_features,param_classify__max_depth,param_classify__min_samples_leaf,mean_test_score,std_test_score,rank_test_score
220,15,500,entropy,sqrt,3,1,0.660417,0.192144,1
16,15,1000,gini,sqrt,None,2,0.658750,0.188064,2
65,20,100,gini,sqrt,3,2,0.657917,0.194632,3
43,15,1000,gini,log2,None,2,0.657083,0.199944,4
295,15,1000,entropy,sqrt,5,4,0.656667,0.202100,5
178,15,1000,entropy,sqrt,None,2,0.655417,0.196987,6
58,15,500,gini,sqrt,3,1,0.655000,0.192072,7
7,15,1000,gini,sqrt,None,1,0.654583,0.197642,8
49,15,500,gini,log2,None,4,0.654583,0.190704,8
256,15,500,entropy,log2,3,2,0.654167,0.201341,10


In [7]:
import pickle as pkl
results_df.to_pickle("results/risultati_GridSearch_rf.pkl")
with open("results/miglior_modello_rf", 'wb') as f:
        pkl.dump(grid.best_estimator_, f)

## Majority rule?

Provo a prendere i 10 migliori?

rf con pca

grafico n_estimators vs score (fold?)